# Chapter 4 — Semantic Search from Scratch

This notebook accompanies **Chapter 4** of *Build an Advanced RAG Application (From Scratch)*.

We start with a corpus of hotel reviews and build a semantic search engine in three increasingly fast forms:

1. **Pure NumPy** — cosine similarity by hand.
2. **NumPy with normalized Euclidean distance** — same ranking, different metric.
3. **FAISS** — the production-grade vector search library.

We finish by comparing FAISS index types (Flat / HNSW / IVF-PQ) on the same query.

> Reusable code lives in `data_loader.py` and `search.py` next to this notebook.


## 1. Setup

Make sure you're in the `advanced-rag` conda env (see the root README) and that the kernel for this notebook points to it.

In [ ]:
import os, sys, time

# macOS: faiss-cpu and torch each bundle their own libomp.dylib; loading both
# in one process causes a silent kernel segfault during FAISS kmeans training
# (e.g. IVF-PQ). KMP_DUPLICATE_LIB_OK silences the abort; OMP_NUM_THREADS=1
# avoids the runtime race. Both must be set before `import torch`.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import torch
import faiss
faiss.omp_set_num_threads(1)

# Make the chapter folder importable
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) if not os.path.exists('search.py') else '.')

from data_loader import load_paris_reviews
from search import (
    load_embedding_model,
    get_embeddings,
    cosine_search,
    euclidean_search,
    build_faiss_cosine_index,
    search_faiss_index,
    build_faiss_indices,
)


## 2. Load the Paris hotel reviews

The dataset is hosted on the HuggingFace Hub. The first call downloads it; subsequent calls hit the local cache.

In [ ]:
df_paris = load_paris_reviews()
print(f"Rows: {len(df_paris):,}")
df_paris.head()

In [ ]:
df_paris.hotel_name.value_counts().head(10)

## 3. Embed the reviews

We use `nomic-ai/nomic-embed-text-v1.5` — a strong open-weight 768-dim embedding model. On CPU this takes a few minutes; on GPU it's much faster.

In [ ]:
model = load_embedding_model()

if torch.cuda.is_available():
    model = model.to("cuda")
    print("CUDA available — model on GPU.")
elif torch.backends.mps.is_available():
    model = model.to("mps")
    print("MPS available — model on Apple Silicon GPU.")
else:
    print("Running on CPU.")

In [ ]:
reviews = df_paris["review_text"].tolist()
review_embeddings = model.encode(reviews, show_progress_bar=True).astype("float32")
print(f"Embeddings shape: {review_embeddings.shape}")

## 4. Search by hand: cosine similarity

No libraries needed. We compute dot products of L2-normalized vectors.

In [ ]:
query = "Hotel with a view of the Eiffel tower."
query_embedding = model.encode([query]).astype("float32")

t0 = time.time()
indices, sims = cosine_search(query_embedding, review_embeddings, k=5)
print(f"Cosine search took {time.time()-t0:.4f}s")

print(f"\nQuery: {query}\n")
for rank, (idx, sim) in enumerate(zip(indices, sims), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (sim={sim:.4f})")
    print(f"   {df_paris.iloc[idx]['review_text'][:200]}...\n")

## 5. Same ranking via normalized Euclidean distance

For unit-norm vectors, `||a − b||² = 2(1 − cos(a, b))`, so Euclidean distance ranks identically to cosine similarity (just inverted).

In [ ]:
indices, distances = euclidean_search(query_embedding, review_embeddings, k=5)
for rank, (idx, dist) in enumerate(zip(indices, distances), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (dist={dist:.4f})")

## 6. Scale up with FAISS

NumPy is fine for a few thousand rows but quickly falls over. FAISS is purpose-built for vector search.

We build an `IndexFlatIP` (inner product) on **L2-normalized** vectors — that's mathematically equivalent to exact cosine similarity but with FAISS's optimized SIMD search.

In [ ]:
faiss_index = build_faiss_cosine_index(review_embeddings)

t0 = time.time()
distances, indices = search_faiss_index(query_embedding, faiss_index, k=5)
print(f"FAISS search took {time.time()-t0:.4f}s\n")

for rank, (idx, sim) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (cos={sim:.4f})")

### Aggregate by hotel

A single hotel may show up many times in the top-k. Group by hotel and rank by mean cosine to surface *places* rather than individual reviews.

In [ ]:
k = 120
distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

hotels = {}
for idx, dist in zip(indices[0], distances[0]):
    name = df_paris.iloc[idx]["hotel_name"]
    h = hotels.setdefault(name, {"reviews": [], "scores": []})
    h["reviews"].append(df_paris.iloc[idx]["review_text"])
    h["scores"].append(float(dist))

ranked = sorted(
    [(n, np.mean(h["scores"]), len(h["reviews"])) for n, h in hotels.items() if len(h["reviews"]) >= 2],
    key=lambda t: t[1],
    reverse=True,
)
for name, mean_score, n in ranked[:10]:
    print(f"{mean_score:.4f}  {n:3d} reviews  {name}")

## 7. Comparing FAISS index types

- **Flat** — exact, full-scan; slow on big corpora.
- **HNSW** — graph-based, fast & accurate, more memory.
- **IVF-PQ** — clustered + quantized, very fast, lossy.

For a small corpus the latency differences are tiny; on millions of vectors they're decisive.

In [ ]:
indices_set = build_faiss_indices(review_embeddings)
k = 100

for name, idx in indices_set.items():
    t0 = time.time()
    distances, top = idx.search(query_embedding, k)
    print(f"{name:6s}: {time.time()-t0:.4f}s  unique hotels = {len({df_paris.iloc[i]['hotel_name'] for i in top[0]})}")

## What's next

Chapter 5 plugs the **decoder** (LLM) on top of these retrievals to start producing grounded answers — the first half of a full RAG pipeline. Chapter 6 wires retrieval + generation together end-to-end and adds a real vector database (Qdrant).